# Chapter 3 — Evidence Before Explanation

**Book alignment:** Debugging AI From First Principles, Chapter 3

**Question this notebook isolates:** One wrong RAG answer, three possible causes (retrieval
omitted the evidence / assembly dropped it / generation contradicted it). Do the three
frozen handoff artifacts — retrieval log, sent context, output — separate them, where the
model's fluent self-explanation cannot?

The "RAG pipeline" here is ~15 lines of dict-shuffling. No LM is called; the point is the
*evidence discipline*, and that is model-agnostic.

In [ ]:
import hashlib

LEDGER = "LEDGER: refund RB status PENDING, no reference number issued"
TICKET = "TICKET 4471: refund timing?"

def h(s):                         # a stand-in 'content hash'
    return hashlib.sha1(s.encode()).hexdigest()[:6]

def pipeline(defect=None):
    # 1. retrieval: keep the ledger + ticket ... unless retrieval drops the ledger
    retrieved = [TICKET] if defect == "retrieval" else [LEDGER, TICKET]
    retrieval_log = [(t.split(":")[0], h(t)) for t in retrieved]

    # 2. assembly: concatenate under a char budget ... a tight budget drops the long ledger line
    budget = 40 if defect == "assembly" else 500
    sent, n = [], 0
    for t in retrieved:
        if n + len(t) <= budget:
            sent.append(t); n += len(t)
    sent = " ".join(sent)

    # 3. generation: answer from context ... unless generation ignores it
    if defect == "generation" or "PENDING" not in sent:
        answer = "Your refund was processed on Sept 2; reference RB-8814."
    else:
        answer = "Your refund is PENDING; no reference number has been issued yet."

    return {"retrieval_log": retrieval_log, "sent_context": sent, "output": answer,
            "explanation": "I retrieved the refund ledger and ticket #4471, which both confirm RB-8814."}

## 1. Three incidents, one wrong answer, one confident (false) explanation

In [ ]:
runs = {d: pipeline(defect=d) for d in ("retrieval", "assembly", "generation")}

for d, r in runs.items():
    print(f"[{d:10}] output: {r['output']}")
assert len({r["output"] for r in runs.values()}) == 1        # identical wrong answer
assert all(r["explanation"] == runs["retrieval"]["explanation"] for r in runs.values())
print("\nsame surface symptom; the model's explanation is byte-identical and cites a ledger line that says the opposite")

## 2. Diff the three frozen artifacts in order — the first ABSENT / CONTRADICTED row convicts

| Hypothesis | retrieval log | sent context | output |
|---|---|---|---|
| retrieval failure | **absent** | absent | wrong |
| assembly failure | present | **absent** | wrong |
| generation failure | present | present | **contradicts context** |

In [ ]:
LEDGER_HASH = h(LEDGER)

def convict(r):
    in_log  = any(hh == LEDGER_HASH for _, hh in r["retrieval_log"])
    in_ctx  = "PENDING" in r["sent_context"]
    contradicted = in_ctx and "processed" in r["output"].lower()
    if not in_log:      return "retrieval boundary"
    if not in_ctx:      return "assembly boundary"
    if contradicted:    return "generation boundary"
    return "no divergence found"

for d, r in runs.items():
    print(f"[{d:10}] ledger in log={any(hh==LEDGER_HASH for _,hh in r['retrieval_log'])}"
          f"  in context={'PENDING' in r['sent_context']}  -> {convict(r)}")

assert convict(runs["retrieval"]) == "retrieval boundary"
assert convict(runs["assembly"]) == "assembly boundary"
assert convict(runs["generation"]) == "generation boundary"
print("\nthree artifacts frozen and diffed in order -> the first break names the boundary and the fix")

## 3. The two-column worksheet: delete the right column, does the diagnosis still stand?

In [ ]:
a = runs["assembly"]
worksheet = {
    "EVIDENCE": [
        f"retrieval log: {a['retrieval_log']}",
        f"ledger hash {LEDGER_HASH} present in retrieval log: {any(hh == LEDGER_HASH for _, hh in a['retrieval_log'])}",
        f"'PENDING' in sent context: {'PENDING' in a['sent_context']}",
    ],
    "EXPLANATION": [
        "model: 'the ledger confirms RB-8814'",
        "engineer guess: 'ledger must have it'",
        "second sample agreed -> 'corroborated'",
    ],
}
for col, rows in worksheet.items():
    print(col)
    for row in rows:
        print("  ", row)

# mechanical test: the assembly diagnosis rests only on EVIDENCE rows
ev = " ".join(worksheet["EVIDENCE"])
assert "present in retrieval log: True" in ev and "'PENDING' in sent context: False" in ev
print("\ndelete EXPLANATION -> 'retrieved but not assembled' still stands on EVIDENCE alone")
print("a diagnosis that collapses without the right column was story-based")

## 4. The load-bearing test: corrupt the explanation, does the behaviour change?

In [ ]:
base = pipeline(defect="generation")
# corrupt the stated reason; the pipeline's answer does not depend on it
corrupted = dict(base, explanation="I retrieved nothing and guessed.")
assert corrupted["output"] == base["output"]
print("explanation corrupted, output unchanged -> the explanation was narration, not a trace")
print("(Lanham's load-bearing test = Ch1's counterfactual, aimed at a stated reason)")

## What we earned

The same wrong refund answer arose at three different boundaries; only the ordered diff of
retrieval log → sent context → output separated them, and each break named a different fix
(raise top-k / fix truncation / constrain generation). The model's explanation was fluent,
cited, coherent, and false — and corrupting it changed nothing, which is the definition of
narration. **Never file a model-generated explanation as an execution trace.**

**Notebook 04 / Chapter 4** gives that first-divergence search a terrain: the debugging
stack, and which layer to rule out first.